# 03 — Evaluation

Runs all baselines and the full HaluGuard pipeline on CodeHaluEval and HumanEval.
Computes hallucination rate, pass rate, and reduction ratios.

**Prerequisites:**
- `data/triplets.jsonl` (from notebook 01)
- `checkpoints/hccs_best.pt` (from notebook 02)

**Methods evaluated:**
1. No context (lower bound)
2. BM25 retrieval
3. Full context (all chunks)
4. CodeBERT similarity (traditional RAG)
5. HaluGuard (HCCS + type router + EFL)

**Runtime:** ~10 hours on T4 for CodeHaluEval test set (~1,500 tasks)

## Setup

In [ ]:
!pip install torch transformers datasets rank-bm25 tqdm --quiet

In [ ]:
import sys
sys.path.insert(0, '..')

from notebooks.utils import mount_drive, check_gpu, load_checkpoint, get_drive_path

my_drive = mount_drive()
DEVICE = check_gpu()
DRIVE_DIR = get_drive_path('HaluGuard')

In [ ]:
import json
from pathlib import Path

import numpy as np
from datasets import load_dataset
from tqdm import tqdm
from transformers import AutoModel, AutoTokenizer

from haluguard.chunker import chunk_repo
from haluguard.efl import execute_code
from haluguard.evaluate import (
    compute_hallucination_rate,
    compute_metrics_table,
    compute_pass_rate,
    compute_per_type_hr,
    compute_reduction_ratio,
)
from haluguard.hccs import HCCSScorer, batch_embed
from haluguard.pipeline import HaluGuardPipeline
from haluguard.type_router import route
from notebooks.utils import append_jsonl, count_jsonl

RESULTS_DIR = DRIVE_DIR / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CKPT_PATH = DRIVE_DIR / 'checkpoints' / 'hccs_best.pt'
print(f'Results dir: {RESULTS_DIR}')
print(f'Checkpoint:  {CKPT_PATH} (exists={CKPT_PATH.exists()})')

## 1. Load Models and Dataset

In [ ]:
# Load CodeHaluEval test split
ds = load_dataset('yuchen814/CodeHalu', split='test')
print(f'Test set: {len(ds)} samples')

In [ ]:
# Load CodeBERT (for HCCS and CodeBERT-similarity baseline)
ENCODER_NAME = 'microsoft/codebert-base'
tokenizer = AutoTokenizer.from_pretrained(ENCODER_NAME)
encoder   = AutoModel.from_pretrained(ENCODER_NAME).to(DEVICE)
encoder.eval()

# Load trained HCCS scorer
scorer = HCCSScorer.load(CKPT_PATH)
scorer = scorer.to(DEVICE)
scorer.eval()

print('Models loaded.')

# TODO: Load code generation LLM (same as notebook 01)
# gen_tokenizer = ...
# gen_model = ...

def generate_code(prompt: str) -> str:
    """TODO: replace with actual LLM call."""
    raise NotImplementedError('Implement generate_code')

## 2. Run Baselines

In [ ]:
def build_prompt(query: str, context_chunks: list) -> str:
    ctx = '\n\n'.join(context_chunks) if context_chunks else '(no context)'
    return f'# Context:\n\n{ctx}\n\n# Task:\n# {query}\n\n# Implementation:\n'


def run_task(query, context_chunks, test_code, task_id, method_name):
    """Run one task with a given context and return a result dict."""
    prompt = build_prompt(query, context_chunks)
    code = generate_code(prompt)
    result = execute_code(code, test_code)
    return {
        'task_id': task_id,
        'method': method_name,
        'passed': result.passed,
        'hallucinated': not result.passed,
        'hallucination_type': result.hallucination_type.value if result.hallucination_type else None,
        'error_type': result.error_type,
    }

In [ ]:
# Baseline 1: No context
NO_CTX_PATH = RESULTS_DIR / 'results_no_context.jsonl'

# TODO: Uncomment to run
# for sample in tqdm(ds, desc='No context'):
#     result = run_task(
#         query=sample['prompt'],
#         context_chunks=[],
#         test_code=sample['test'],
#         task_id=sample['task_id'],
#         method_name='no_context',
#     )
#     append_jsonl(result, NO_CTX_PATH)

print(f'No-context results: {count_jsonl(NO_CTX_PATH)} samples')

In [ ]:
# Baseline 2: BM25 retrieval
from rank_bm25 import BM25Okapi

BM25_PATH = RESULTS_DIR / 'results_bm25.jsonl'

def bm25_select(query: str, chunks: list, top_k: int = 5) -> list:
    """Select top-k chunks by BM25 keyword similarity."""
    tokenized_corpus = [c.lower().split() for c in chunks]
    bm25 = BM25Okapi(tokenized_corpus)
    scores = bm25.get_scores(query.lower().split())
    top_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]
    return [chunks[i] for i in top_indices]


# TODO: Uncomment to run
# for sample in tqdm(ds, desc='BM25'):
#     chunks = chunk_repo(sample['context'])  # adjust key to dataset schema
#     selected = bm25_select(sample['prompt'], chunks)
#     result = run_task(sample['prompt'], selected, sample['test'], sample['task_id'], 'bm25')
#     append_jsonl(result, BM25_PATH)

print(f'BM25 results: {count_jsonl(BM25_PATH)} samples')

In [ ]:
# Baseline 3: Full context (all chunks)
FULL_CTX_PATH = RESULTS_DIR / 'results_full_context.jsonl'

# TODO: Uncomment to run
# for sample in tqdm(ds, desc='Full context'):
#     chunks = chunk_repo(sample['context'])
#     result = run_task(sample['prompt'], chunks, sample['test'], sample['task_id'], 'full_context')
#     append_jsonl(result, FULL_CTX_PATH)

print(f'Full-context results: {count_jsonl(FULL_CTX_PATH)} samples')

In [ ]:
# Baseline 4: CodeBERT cosine similarity (traditional RAG)
import torch

CODEBERT_PATH = RESULTS_DIR / 'results_codebert_similarity.jsonl'

def cosine_select(query: str, chunks: list, top_k: int = 5) -> list:
    """Select top-k chunks by cosine similarity to query embedding."""
    from haluguard.hccs import embed_code
    q_emb = embed_code(query, tokenizer, encoder, device=DEVICE)  # (768,)
    c_embs = batch_embed(chunks, tokenizer, encoder, device=DEVICE)  # (N, 768)
    # Cosine similarity: dot product of L2-normalised vectors
    q_norm = q_emb / (np.linalg.norm(q_emb) + 1e-8)
    c_norms = c_embs / (np.linalg.norm(c_embs, axis=1, keepdims=True) + 1e-8)
    sims = c_norms @ q_norm  # (N,)
    top_indices = np.argsort(sims)[::-1][:top_k]
    return [chunks[i] for i in top_indices]


# TODO: Uncomment to run
# for sample in tqdm(ds, desc='CodeBERT similarity'):
#     chunks = chunk_repo(sample['context'])
#     selected = cosine_select(sample['prompt'], chunks)
#     result = run_task(sample['prompt'], selected, sample['test'], sample['task_id'], 'codebert_sim')
#     append_jsonl(result, CODEBERT_PATH)

print(f'CodeBERT similarity results: {count_jsonl(CODEBERT_PATH)} samples')

In [ ]:
# HaluGuard: Full pipeline (HCCS + type router + EFL)
HALUGUARD_PATH = RESULTS_DIR / 'results_haluguard.jsonl'

pipeline = HaluGuardPipeline(
    scorer=scorer,
    tokenizer=tokenizer,
    encoder=encoder,
    top_k=5,
    device=DEVICE,
)

# TODO: Uncomment to run
# for sample in tqdm(ds, desc='HaluGuard'):
#     result = pipeline.run(
#         query=sample['prompt'],
#         repo_files=sample['context'],
#         test_code=sample['test'],
#         generate_fn=generate_code,
#         max_iterations=3,
#     )
#     record = {
#         'task_id': sample['task_id'],
#         'method': 'haluguard',
#         'passed': result['passed'],
#         'hallucinated': not result['passed'],
#         'hallucination_type': (
#             result['history'][-1].hallucination_type.value
#             if result['history'] and result['history'][-1].hallucination_type
#             else None
#         ),
#         'iterations': result['iterations'],
#     }
#     append_jsonl(record, HALUGUARD_PATH)

print(f'HaluGuard results: {count_jsonl(HALUGUARD_PATH)} samples')

## 3. Compute and Display Results

In [ ]:
def load_jsonl(path: Path) -> list:
    if not path.exists():
        return []
    with path.open() as f:
        return [json.loads(line) for line in f if line.strip()]


results_by_method = {
    'no_context':        load_jsonl(NO_CTX_PATH),
    'bm25':              load_jsonl(BM25_PATH),
    'full_context':      load_jsonl(FULL_CTX_PATH),
    'codebert_sim':      load_jsonl(CODEBERT_PATH),
    'haluguard':         load_jsonl(HALUGUARD_PATH),
}

# Remove empty methods
results_by_method = {k: v for k, v in results_by_method.items() if v}

if results_by_method:
    table = compute_metrics_table(results_by_method, baseline_key='no_context')
    print(f'{'Method':<22} {'HR':>6}  {'Pass%':>6}  {'Reduction':>9}')
    print('-' * 50)
    for row in table:
        print(
            f'{row["method"]:<22}'
            f'{row["hr"]:>6.3f}'
            f'  {row["pass_rate"]:>6.3f}'
            f'  {row["reduction_ratio"]:>9.3f}'
        )
else:
    print('No results found. Run evaluation cells above first.')

In [ ]:
# Per-type hallucination rates for HaluGuard vs BM25 baseline
if 'haluguard' in results_by_method and 'bm25' in results_by_method:
    print('Per-type hallucination rate:')
    print(f'{'Type':<12} {'BM25':>8}  {'HaluGuard':>10}  {'Reduction':>10}')
    print('-' * 45)
    bm25_by_type = compute_per_type_hr(results_by_method['bm25'])
    hg_by_type   = compute_per_type_hr(results_by_method['haluguard'])
    for t in ['resource', 'naming', 'mapping', 'logic']:
        bm25_hr = bm25_by_type.get(t, float('nan'))
        hg_hr   = hg_by_type.get(t, float('nan'))
        try:
            reduction = compute_reduction_ratio(bm25_hr, hg_hr)
        except (ValueError, ZeroDivisionError):
            reduction = float('nan')
        print(f'{t:<12} {bm25_hr:>8.3f}  {hg_hr:>10.3f}  {reduction:>10.3f}')
else:
    print('Run BM25 and HaluGuard evaluations first.')

## 4. Ablation Study

Test each component in isolation to quantify its individual contribution:
1. HCCS only (no router, no EFL)
2. HCCS + router (no EFL)
3. HCCS + EFL (no router)
4. Full system (HCCS + router + EFL)

In [ ]:
# TODO: Implement ablation runs
# Each variant is a simplified version of the pipeline.run() loop:
#   - HCCS only: use scorer.score_chunks, no router, no EFL (1 iteration)
#   - HCCS + router: add type_router.route on the initial query
#   - HCCS + EFL: use EFL but skip router (no pre-generation routing)
#   - Full: pipeline.run() as implemented in the cell above

print('Ablation cells — implement after running full evaluation.')